***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.x 延伸阅读与参考文献](2_x_further_reading_and_references.ipynb)
    * 下一节：[第 3 章：位置天文学](../3_Positional_Astronomy/3_0_introduction.ipynb)

***


导入标准模块：

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from IPython.display import HTML 
#HTML('../style/course.css') #apply general CSS
HTML('../style/code_toggle.html')

导入部分特定模块：

In [ ]:
pass

## 2.y. 练习<a id='math:sec:exercises'></a><!--\label{math:sec:exercises}-->

下面给出一组适用于干涉测量课程的小练习。它们并不是与主线脱节的“纯数学题”，而是围绕第 2 章最核心的几种操作展开：函数表示、卷积、傅里叶变换以及由图像直观回到解析表达式。

建议的使用方式是：先独立完成题目，再对照后面的示例答案检查自己的推导结构。示例答案的目的主要是展示一条清晰、可复查的解题路径，而不是唯一写法。若你能够用另一套等价推导得到相同结果，同样是完全合理的。


这些练习也可以当作进入后续章节前的自检表。若你已经能比较熟练地完成这里的题目，那么再去读采样、可见度、成像和去卷积时，就更容易把抽象符号与实际算法联系起来。


### 2.y.1. 傅里叶变换与卷积：三角函数的傅里叶变换<a id='math:sec:exercises_fourier_triangle'></a><!--\label{math:sec:exercises_fourier_triangle}-->

考虑下图所示的三角函数。

In [ ]:
def plotviewgraph(fig, ax, xmin = 0, xmax = 1., ymin = 0., ymax = 1.):
    """
    Prepare a viewvgraph for plotting a function
    
    Parameters:
    fig:          Matplotlib figure
    ax:           Matplotlib subplot
    xmin (float): Minimum of range
    xmax (float): Maximum of range
    ymin (float): Minimum of function
    ymax (float): Maximum of function

    return: axis and vertical and horizontal tick length
    """
    
    # Axis ranges
    ax.axis([xmin-0.1*(xmax-xmin), xmax+0.1*(xmax-xmin), -0.2*(ymax-ymin), ymax])
    ax.axis('off')

    # get width and height of axes object to compute, see https://3diagramsperpage.wordpress.com/2014/05/25/arrowheads-for-axis-in-matplotlib/

    # matching arrowhead length and width
    dps = fig.dpi_scale_trans.inverted()
    bbox = ax.get_window_extent().transformed(dps)
    width, height = bbox.width, bbox.height
 
    # manual arrowhead width and length
    hw = 1./15.*(ymax-ymin) 
    hl = 1./30.*(xmax-xmin)
    lw = 1. # axis line width
    ohg = 0.3 # arrow overhang
 
    # compute matching arrowhead length and width
    yhw = hw/(ymax-ymin)*(xmax-xmin)* height/width 
    yhl = hl/(xmax-xmin)*(ymax-ymin)* width/height

    # Draw arrows
    ax.arrow(xmin-0.1*(xmax-xmin),0, 1.2*(xmax-xmin),0, fc='k', ec='k', lw = lw, 
         head_width=hw, head_length=hl, overhang = ohg, 
         length_includes_head= True, clip_on = False)
    ax.arrow(0,ymin-0.1*(ymax-ymin), 0., 1.4*(ymax-ymin), fc='k', ec='k', lw = lw, 
         head_width=yhw, head_length=yhl, overhang = ohg, 
         length_includes_head= True, clip_on = False)
    
    # Draw ticks for A, -A, and B
    twv = 0.01*height # vertical tick width
    twh = twv*(xmax-xmin)/(ymax-ymin)/ width*height
        
    return twv, twh

def plottriangle():
    
    A = 1.
    B = 1.
    
    # Start the plot, create a figure instance and a subplot
    fig = plt.figure(figsize=(20,5))
    ax  = fig.add_subplot(111)
    
    twv, twh = plotviewgraph(fig, ax, xmin = -A, xmax = A, ymin = 0., ymax = B)
    
    ticx = [[-A,'-A'],[A,'A']]
    
    for tupel in ticx:
        ax.plot([tupel[0],tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0], 0.-twh, tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')
    
    ticy = [[B,'B']]
    for tupel in ticy:
        ax.plot([-twh, twh], [tupel[0], tupel[0]], 'k-')
        ax.text(0.+twv, tupel[0], tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'bottom', color = 'black')

    
    # Plot the function
    ax.plot([-A,0.,A],[0., B, 0.], 'r-', lw = 2)

    # Annotate axes
    ax.text(0.-twh, 1.2*(B), r'$f(x)$', fontsize = 24, horizontalalignment = 'right', verticalalignment = 'bottom', color = 'black')
    ax.text(1.2*B, 0., r'$x$', fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')

    
    # Show amplitude
#    plt.annotate(s='', xy=(mu+2*sigma,0.), xytext=(mu+2*sigma,a), \
#                 arrowprops=dict(color = 'magenta', arrowstyle='<->'))
#    ax.text(mu+2*sigma+sigma/10., a/2, '$a$', fontsize = 12, horizontalalignment = 'left', \
#            verticalalignment = 'center', color = 'magenta')

    
plottriangle()
# <a id='math:fig:triangle'></a><!--\label{math:fig:triangle}-->

**图 2.y.1：** 宽度为 $2A$、幅度为 $B$ 的三角函数。<a id='math:fig:triangle'></a><!--\label{math:fig:triangle}-->

<b>题目：</b>
<ol type="A">
  <li>利用函数的对称性，你能判断 $f$ 的傅里叶变换的复数部分具有怎样的性质吗？</li>
  <li>把函数 $f$ 用两种方式写出来：一种写成分段定义函数，另一种写成矩形函数与其自身的卷积。</li>
  <li>把 $f$ 写成 方窗（矩形）函数与其自身的卷积，并利用卷积定理计算其傅里叶变换。</li>
</ol>

#### 2.y.1.1 三角函数的傅里叶变换：题目 1 的示例答案。<a id='math:sec:exercises_fourier_triangle_a'></a><!--\label{math:sec:exercises_fourier_triangle_a}-->

<b>利用函数的对称性，你能判断 $f$ 的傅里叶变换的复数部分以及对称性吗？</b>

该函数是实值函数（$f^*(x)\,=\,f(x)$），同时又是偶函数（$f(x)\,=\,f(-x)$），因此它是厄米函数（$f^*(x)\,=\,f(-x)$，定义见[这里 &#10142;](2_4_the_fourier_transform.ipynb#math:sec:fourier_transforms_of_real_valued_and_hermetian_functions) <!--\ref{math:sec:fourier_transforms_of_real_valued_and_hermetian_functions}-->）。根据 [第 2.4.6 节 &#10142;](2_4_the_fourier_transform.ipynb#math:sec:fourier_transforms_of_real_valued_and_hermetian_functions)<!--\ref{math:sec:fourier_transforms_of_real_valued_and_hermetian_functions}-->, 这意味着它的傅里叶变换是<b>实值</b>函数（因为它是厄米函数的傅里叶变换），同时也仍然是厄米函数（因为它是实值函数的傅里叶变换）。因此它也一定是<b>偶函数</b>（$f^*(x)\,=\,f(x) \,\land\, f^*(x)\,=\,f(-x)\,\Rightarrow\,f(x)\,=\,f(-x)$）。所谓实值，意味着 $f$ 的复数部分为 $0$。

#### 2.y.1.2 三角函数的傅里叶变换：题目 2 的示例答案。<a id='math:sec:exercises_fourier_triangle_b'></a><!--\label{math:sec:exercises_fourier_triangle_b}-->

<b>把函数 $f$ 用两种方式写出来：一种写成分段定义函数，另一种写成矩形函数与其自身的卷积。</b>

第一部分比较直接：

<a id='math:eq:y_001'></a><!--\label{math:eq:y_001}-->$$
\begin{align*}
f(x)   &= \left \{
     \begin{array}{lll}
    B-\frac{B}{A}|x| & {\rm for} & |x| \leq A\\
    0 & {\rm for} & |x| > A
\end{array}\right .
\end{align*}
$$

第二部分稍微需要一点计算，但过程并不复杂。

<a id='math:eq:y_002'></a><!--\label{math:eq:y_002}-->
$$
\begin{align*}
    f(x)   \,&=\,\frac{B}{A}\cdot  \Pi_{-\frac{A}{2},\frac{A}{2}}\circ \Pi_{-\frac{A}{2},\frac{A}{2}}(x)\\
&=\,\frac{B}{A}\cdot\Pi_A\circ \Pi_A\,\,\, {\rm，其中} \,\,\,\Pi_A(x) \,=\,\Pi(\frac{x}{A})\\
\end{align*}
$$

利用 [方窗函数的定义 &#10142;](2_2_important_functions.ipynb#math:sec:boxcar_and_rectangle_function) <!--\ref{math:sec:boxcar_and_rectangle_function}--> 和 [卷积的定义 &#10142;](2_5_convolution.ipynb#math:sec:definition_of_the_convolution) <!--\ref{math:sec:definition_of_the_convolution}-->，可以看出：

<a id='math:eq:y_003'></a><!--\label{math:eq:y_003}-->
$$
\begin{align*}
\Pi_{-\frac{A}{2},\frac{A}{2}}\circ \Pi_{-\frac{A}{2},\frac{A}{2}}(x)\,& =\,  \int_{-\infty}^{\infty}\Pi_{-\frac{A}{2},\frac{A}{2}}(t)\Pi_{-\frac{A}{2},\frac{A}{2}}(x-t)\,dt\\
& =\,  \int_{-\frac{A}{2}}^{\frac{A}{2}}\Pi_{-\frac{A}{2},\frac{A}{2}}(x-t)\,dt\\
& \underset{u\,=\,x-t}{=} \, \int_{u(-\frac{A}{2})}^{u(\frac{A}{2})}\Pi_{-\frac{A}{2},\frac{A}{2}}(u)\frac{dx}{du}\,du\\
& =\, \int_{x+\frac{A}{2}}^{x-\frac{A}{2}}\Pi_{-\frac{A}{2},\frac{A}{2}}(u)\cdot(-1)du\\
& =\, \int_{x-\frac{A}{2}}^{x+\frac{A}{2}}\Pi_{-\frac{A}{2},\frac{A}{2}}(u)du\\
\end{align*}
$$

因此，

<a id='math:eq:y_004'></a><!--\label{math:eq:y_004}-->
\begin{align*}
|x| \,>\, A \,&\Rightarrow\,\Pi_{-\frac{A}{2},\frac{A}{2}}\circ \Pi_{-\frac{A}{2},\frac{A}{2}}(x)\, =\, 0\\
-A\,\leq\,x\,\leq 0\,&\Rightarrow \,\Pi_{-\frac{A}{2},\frac{A}{2}}\circ \Pi_{-\frac{A}{2},\frac{A}{2}}(x)\,=\,\int_{-\frac{A}{2}}^{x+\frac{A}{2}}du\,=\,A+x\\
0\,\leq\,x\,\leq A\,&\Rightarrow \,\Pi_{\frac{A}{2},\frac{A}{2}}\circ \Pi_{-\frac{A}{2},\frac{A}{2}}(x)\,=\,\int_{x-\frac{A}{2}}^{\frac{A}{2}}du\,=\,A-x\\
\end{align*}

这与[上面的分段定义 &#10549;](2_y_exercises.ipynb#math:sec:math:eq:y_001)完全一致。

#### 2.y.1.3 三角函数的傅里叶变换：题目 3 的示例答案。<a id='math:sec:exercises_fourier_triangle_c'></a><!--\label{math:sec:exercises_fourier_triangle_c}-->

我们知道（[卷积定理 &#10142;](2_7_fourier_theorems.ipynb#math:sec:convolution_theorem)<!--\ref{math:sec:convolution_theorem}-->, [相似定理 &#10142;](2_7_fourier_theorems.ipynb#math:sec:similarity_theorem)<!--\ref{math:sec:similarity_theorem}-->, [三角函数的定义 &#10549;](#math:eq:y_002)<!--\ref{math:eq:y_002}-->, [矩形方窗函数的傅里叶变换 &#10142;](2_4_the_fourier_transform.ipynb#math:sec:fourier_transform_of_the_rectangle_and_the_sinc_function)<!--\ref{math:sec:convolution_theorem}-->）：

<a id='math:eq:y_005'></a><!--\label{math:eq:y_005}-->$$
\begin{align*}
\mathscr{F}\{h\circ g\}\,&=\,\mathscr{F}\{h\}\cdot\mathscr{F}\{g\}\\
g\,=\,h(ax) \,&\Rightarrow\, \mathscr{F}\{g\}(s) = \frac{1}{|a|}\mathscr{F}\{h\}(\frac{s}{a})\\
f(x) \,&=\, \frac{B}{A}\Pi_A\circ\Pi_A(x)\\
\Pi_A(x)\,&=\,\Pi(\frac{x}{A})\\
\mathscr{F}\{\Pi\}(s) \,&=\,{\rm sinc}(s) \\
\end{align*}
$$

这样一来，计算会简短得多。

<a id='math:eq:y_006'></a><!--\label{math:eq:y_006}-->$$
\begin{align*}
\mathscr{F}\{f\}(s)\,&=\,\mathscr{F}\{\frac{B}{A}\Pi_A\circ\Pi_A\}(s)\\
&=\,\frac{B}{A}\mathscr{F}\{\Pi_A\}(s)\cdot\mathscr{F}\{\Pi_A\}(s)\\
&=\,\frac{B}{A}\mathscr{F}\{A\Pi\}(As)\cdot\mathscr{F}\{A\Pi_A\}(As)\\
&=\,AB\,\mathscr{F}\{\Pi\}(As)\cdot\mathscr{F}\{\Pi\}(As)\\
&=\,AB\,{\rm sinc}(As)\cdot{\rm sinc}(As)\\
&=\,AB\,{\rm sinc}^2(As)\\
&=\,AB\,\frac{sin^2 A\pi s}{A^2\pi^2 s^2}\\
\end{align*}$$

因此，解如下所示：

In [ ]:
def plotfftriangle():
    
    A = 1.
    B = 1.
    
    # Start the plot, create a figure instance and a subplot
    fig = plt.figure(figsize=(20,5))
    ax  = fig.add_subplot(111)

    
    twv, twh = plotviewgraph(fig, ax, xmin = -3./A, xmax = 3./A, ymin = -0.3, ymax = B)    
    ticx = [[-A,r'$-\frac{1}{A}$'],[A,'A']]
    
    ticx = [[-3.*A, r'$\frac{-3}{A}$'], [-2.*A, r'$\frac{-2}{A}$'], [-1./A, r'$\frac{-1}{A}$'], [1./A, r'$\frac{1}{A}$'], [2./A, r'$\frac{2}{A}$'], [3./A, r'$\frac{3}{A}$']]
    for tupel in ticx:
        ax.plot([tupel[0],tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0], 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'center', verticalalignment = 'top', color = 'black')
    
    ticx = [[0.,r'$0$']]
    for tupel in ticx:
        ax.plot([tupel[0],tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0]+twh, 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')

    
    ticy = [[B,r'$\frac{B}{A}$']]
    for tupel in ticy:
        ax.plot([-twh, twh], [tupel[0], tupel[0]], 'k-')
        ax.text(0.+twv, tupel[0], tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'bottom', color = 'black')

    # Plot the function
    x = np.linspace(-4.*A, 4.*A, 900)
    y = np.power(np.sinc(x),2)

    # Annotate axes
    ax.text(0.-A/20, 1.2*(B), r'$f(x)$', fontsize = 24, horizontalalignment = 'right', verticalalignment = 'bottom', color = 'black')
    ax.text(1.2*3.*A, 0., r'$x$', fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')
    
    ax.plot(x, y, 'r-', lw = 2)
        
plotfftriangle()
# <a id='math:fig:fftriangle'></a><!--\label{math:fig:fftriangle}-->

**图 2.y.2：** 三角函数的傅里叶变换。<a id='math:fig:ft_of_triangle'></a><!--\label{math:fig:ft_of_triangle}-->

### 2.y.2. 傅里叶变换与卷积：两个有限支撑函数的卷积<a id='math:sec:exercises_convolution_of_two_functions_with_finite_support'></a><!--\label{math:sec:exercises_convolution_of_two_functions_with_finite_support}-->

考虑下图给出的两个函数：

In [ ]:
def plotrectntria():
    
    A = 1.
    B = 1.4
    
    # Start the plot, create a figure instance and a subplot
    fig = plt.figure(figsize=(20,5))
    ax  = fig.add_subplot(121)
    
    twv, twh = plotviewgraph(fig, ax, xmin = 0., xmax = 3.*A, ymin = 0., ymax = 3.)    

    ticx = [[1.*A, r'$A$'], [2.*A, r'$2A$'], [3.*A, r'$3A$']]
    for tupel in ticx:
        ax.plot([tupel[0],tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0], 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'center', verticalalignment = 'top', color = 'black')
    
    ticx = [[0.,r'$0$']]
    for tupel in ticx:
        ax.plot([-tupel[0],-tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0]+twh, 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')

    
    ticy = [[1,r'$1$'], [2.,r'$2$'], [3.,r'$3$']]
    for tupel in ticy:
        ax.plot([-twh, twh], [tupel[0], tupel[0]], 'k-')
        ax.text(0.-twv, tupel[0], tupel[1], fontsize = 24, horizontalalignment = 'right', verticalalignment = 'center', color = 'black')

    ticy = [[B, r'$B$']]
    for tupel in ticy:
        ax.plot([-twh, twh], [tupel[0], tupel[0]], 'k-')
        ax.text(0.+twv, tupel[0], tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'bottom', color = 'black')

        # Plot the function
    x = [A, A, 2*A, 2*A]
    y = [0., B, B, 0.]
    ax.plot(x, y, 'r-', lw = 2)

    x = [0., A]
    y = [B, B]
    ax.plot(x, y, 'k--', lw = 1)

    # Annotate axes
    ax.text(0.-3.*twh, 1.2*3., r'$g(x)$', fontsize = 24, horizontalalignment = 'right', verticalalignment = 'bottom', color = 'black')
    ax.text(1.1*3.*A, 0., r'$x$', fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')
    
    ###################
    
    ax  = fig.add_subplot(122)

    twv, twh = plotviewgraph(fig, ax, xmin = 0., xmax = 3.*A, ymin = 0., ymax = 3.)    

    ticx = [[1.*A, r'$A$'], [2.*A, r'$2A$'], [3.*A, r'$3A$']]
    for tupel in ticx:
        ax.plot([tupel[0],tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0], 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'center', verticalalignment = 'top', color = 'black')
    
    ticx = [[0.,r'$0$']]
    for tupel in ticx:
        ax.plot([-tupel[0],-tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0]+twh, 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')

    
    ticy = [[1,r'$1$'], [2.,r'$2$'], [3.,r'$3$']]
    for tupel in ticy:
        ax.plot([-twh, twh], [tupel[0], tupel[0]], 'k-')
        ax.text(0.-twv, tupel[0], tupel[1], fontsize = 24, horizontalalignment = 'right', verticalalignment = 'center', color = 'black')


        # Plot the function
    x = [A, A, 2*A, 3*A, 3*A]
    y = [0., 1., 3., 1., 0.]
    ax.plot(x, y, 'r-', lw = 2)

    x = [0., A]
    y = [1., 1.]
    ax.plot(x, y, 'k--', lw = 1)

    x = [0., 2*A]
    y = [3., 3.]
    ax.plot(x, y, 'k--', lw = 1)

    # Annotate axes
    ax.text(0.-3.*twh, 1.2*3., r'$f(x)$', fontsize = 24, horizontalalignment = 'right', verticalalignment = 'bottom', color = 'black')
    ax.text(1.1*3.*A, 0., r'$x$', fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')

plotrectntria()
# <a id='math:fig:two_fs_with_finite_support'></a><!--\label{math:fig:two_fs_with_finite_support}-->

**图 2.y.3：** 两个具有有限支撑的函数 $g$ 和 $h$。<a id='math:fig:two_fs_with_finite_support'></a><!--\label{math:fig:two_fs_with_finite_support}-->

<b>题目：</b>
<ol type="A">
  <li>写出函数 $g$ 和 $h$。</li>
  <li>计算它们的卷积。</li>
</ol>

#### 2.y.2.1 两个有限支撑函数的卷积：题目 1 的示例答案。<a id='math:sec:exercises_convolution_of_two_functions_with_finite_support_a'></a><!--\label{math:sec:exercises_convolution_of_two_functions_with_finite_support_a}-->

<b>写出函数 $g$ 和 $h$。</b>

<a id='math:eq:y_007'></a><!--\label{math:eq:y_007}-->$$
\begin{align*}
h(x)   &= \left \{
     \begin{array}{lll}
    B & {\rm for} & A \leq x \leq 2A\\
    0 & {\rm else}
\end{array}\right .\\
g(x)   &= \left \{
     \begin{array}{lll}
    g_1(x)\,=\,\frac{2}{A}\left(x-\frac{A}{2}\right) & {\rm for} & A \leq x \leq 2A\\
    g_2(x)\,=\,-\frac{2}{A}\left(x-\frac{7A}{2}\right) & {\rm for} & 2A \leq x \leq 3A\\
    0 & {\rm else}
\end{array}\right .\\
\end{align*}
$$

#### 2.y.2.2 两个有限支撑函数的卷积：题目 2 的示例答案。<a id='math:sec:exercises_convolution_of_two_functions_with_finite_support_b'></a><!--\label{math:sec:exercises_convolution_of_two_functions_with_finite_support_b}-->

我们需要计算下面的积分（见 [卷积的定义 &#10142;](2_5_convolution.ipynb#math:sec:definition_of_the_convolution) <!--\ref{math:sec:definition_of_the_convolution}-->）：

<a id='math:eq:y_008'></a><!--\label{math:eq:y_008}-->$$
g\circ h(x) \, = \, \int_{-\infty}^{\infty}g(x-t)h(t)\,dt
$$

为此，需要根据 $g(x-t)$ 与 $h(t)$ 的支撑集（即函数非零的区间），分别在不同的 $x$ 范围内计算该积分。

为了方便，先把[上面的函数 &#10549;](#math:eq:y_008)<!--\ref{math:eq:y_008}-->改写为：

<a id='math:eq:y_009'></a><!--\label{math:eq:y_009}-->$$
\begin{align*}
g(x-t)   &= \left \{
     \begin{array}{lll}
    B & {\rm for} & -2A+x \leq t \leq -A+x\\
    0 & {\rm else}
\end{array}\right .\\
h(t)   &= \left \{
     \begin{array}{lll}
    h_1(t)\,=\,\frac{2}{A}\left(t-\frac{A}{2}\right) & {\rm for} & A \leq t \leq 2A\\
    h_2(t)\,=\,-\frac{2}{A}\left(t-\frac{7A}{2}\right) & {\rm for} & 2A \leq t \leq 3A\\
    0 & {\rm else}
\end{array}\right .\\
\end{align*}
$$

情形 1：

<a id='math:eq:y_010'></a><!--\label{math:eq:y_010}-->$$
\begin{align*}
x \,&<\, 2A\qquad\,\Rightarrow\\
g\circ h(x) \, &= \, \int_{-\infty}^{A}g(x-t)h(t)\,dt\\
&=\, 0
\end{align*}
$$

情形 2：

<a id='math:eq:y_011'></a><!--\label{math:eq:y_011}-->$$
\begin{align*}
2A \,&\leq x \,<\, 3A\qquad\Rightarrow\\
g\circ h(x) \, &= \, \int_{-\infty}^{\infty}g(x-t)h(t)\,dt\\
&=\, \int_{A}^{x-A}B\,h_1(t)\,dt\,\\
&=\,\int_{A}^{x-A}\frac{2B}{A}\left(t-\frac{A}{2}\right)\,dt\\
&=\,\frac{B}{A}\left(x^2-3Ax+2A^2\right)\\
\end{align*}$$

情形 3：

<a id='math:eq:y_012'></a><!--\label{math:eq:y_012}-->$$
\begin{align*}
3A \,&\leq\, x \,<\, 4A\qquad\Rightarrow\\
g\circ h(x) \, &=\, \int_{x-2A}^{2A}B\,h_1(t)\,dt+ \int_{2A}^{x-A}B\,h_2(t)\,dt\\
&=\,\int_{x-2A}^{2A}\frac{2B}{A}\left(t-\frac{A}{2}\right)\,dt- \int_{2A}^{x-A}\frac{2B}{A}\left(t-\frac{7A}{2}\right)\,dt\\
&=\,\frac{B}{A}\left(-2x^2+14Ax-22A^2\right)\\
\end{align*}
$$

情形 4：

<a id='math:eq:y_013'></a><!--\label{math:eq:y_013}-->$$
\begin{align*}
4A \,&\leq x \,<\, 5A\qquad\Rightarrow\\
g\circ h(x) \, &=\, \int_{x-2A}^{3A}B\,h_2(t)\,dt\,=\,\int_{x-2A}^{3A}-\frac{2B}{A}\left(t-\frac{7A}{2}\right)\,dt\\
&=\,\frac{B}{A}\left(x^2-11Ax+30A^2\right)\\
\end{align*}
$$

情形 5：

<a id='math:eq:y_014'></a><!--\label{math:eq:y_014}-->$$
\begin{align*}
5A&\,\leq\,x\qquad\,\Rightarrow\\
g\circ h(x) \, &= \, \int_{3A}^{\infty}g(x-t)h(t)\,dt\\
&=\, 0
\end{align*}
$$

综上，$g$ 与 $h$ 的卷积得到如下分段函数：

<a id='math:eq:y_014'></a><!--\label{math:eq:y_014}-->$$
\begin{align*}
g\circ h(x) \, &=      
 \frac{B}{A}\left\{\begin{array}{lll}
    0 & {\rm for} & x < 2A \\
   x^2-3Ax+2A^2 & {\rm for} & 2A \leq x < 3A\\
   -2x^2+14Ax-22A^2 & {\rm for} & 3A \leq x < 4A\\
   x^2-11Ax+30A^2 & {\rm for} & 4A \leq x < 5A\\
    0 & {\rm for} & 5A \leq x \\
\end{array}\right .\\
\end{align*}$$

In [ ]:
def rectntriaconv(A,B,x):
    
    xn = x[x < (2*A)]
    yn = xn*0.
    y = yn
    
    xn = x[(x == 2*A) | (x > 2*A) & (x < 3*A)]
    yn = (B/A)*(np.power(xn,2)-3*A*xn+2*np.power(A,2))
    y = np.append(y,yn)
        
    xn = x[(x == 3*A) | (x > 3*A) & (x < 4*A)]
    yn = (B/A)*((-2*np.power(xn,2))+14*A*xn-22*np.power(A,2))
    y = np.append(y,yn)
        
    xn = x[(x == 4*A) | (x > 4*A) & (x < 5*A)]
    yn = (B/A)*(np.power(xn,2)-11*A*xn+30*np.power(A,2))
    y = np.append(y,yn)
        
    xn = x[(x == 5*A) | (x > 5*A)]
    yn = xn*0.
    y = np.append(y,yn)

    return y

def plotrectntriaconv():
    A = 1.
    B = 1.4
    
    # Start the plot, create a figure instance and a subplot
    fig = plt.figure(figsize=(20,5))
    ax  = fig.add_subplot(121)
    
    twv, twh = plotviewgraph(fig, ax, xmin = 0., xmax = 6.*A, ymin = 0., ymax = 2.5*A*B)    

    ticx = [[1.*A, r'$A$'], [2.*A, r'$2A$'], [3.*A, r'$3A$'], [4.*A, r'$4A$'], [5.*A, r'$5A$'], [6.*A, r'$6A$']]
    for tupel in ticx:
        ax.plot([tupel[0],tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0], 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'center', verticalalignment = 'top', color = 'black')
    
    ticx = [[0.,r'$0$']]
    for tupel in ticx:
        ax.plot([-tupel[0],-tupel[0]],[-twv, twv], 'k-')
        ax.text(tupel[0]+twh, 0.-2.*twh, tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')
    
    ticy = [[2*A*B, r'$2AB$'], [2.5*A*B, r'$\frac{5}{2}AB$']]
    for tupel in ticy:
        ax.plot([-twh, twh], [tupel[0], tupel[0]], 'k-')
        ax.text(0.+5*twv, tupel[0], tupel[1], fontsize = 24, horizontalalignment = 'left', verticalalignment = 'bottom', color = 'black')

    # Plot the function
    x = np.linspace(0., 7.*A, 900)
    y = rectntriaconv(A,B,x)
    ax.plot(x, y, 'r-', lw = 2)

    # Plot a few lines
    x = [0., 4*A]
    y = [2.*A*B, 2.*A*B]
    ax.plot(x, y, 'k--', lw = 1)

    x = [0., 3.5*A]
    y = [2.5*A*B, 2.5*A*B]
    ax.plot(x, y, 'k--', lw = 1)

    x = [3.*A, 3.*A]
    y = [0., 2.*A*B]
    ax.plot(x, y, 'k--', lw = 1)

    x = [4.*A, 4.*A]
    y = [0., 2.*A*B]
    ax.plot(x, y, 'k--', lw = 1)

    
    # Annotate axes
    ax.text(0.-3.*twh, 1.25*2.5*A*B, r'$g\circ h(x)$', fontsize = 24, horizontalalignment = 'right', verticalalignment = 'bottom', color = 'black')
    ax.text(1.1*6.*A, 0., r'$x$', fontsize = 24, horizontalalignment = 'left', verticalalignment = 'top', color = 'black')

plotrectntriaconv()
# <a id='math:fig:two_fs_wfs'></a><!--\label{math:fig:two_fs_wfs}-->

**图 2.y.4：** 图 2.y.3 中两个函数 $g$ 与 $h$ 的卷积。<!--\ref{math:fig:two_fs_with_finite_support}-->

***

* 下一节：[第 3 章：位置天文学](../3_Positional_Astronomy/3_0_introduction.ipynb)